# Efficient LLM Adaptation with LoRA

This notebook demonstrates parameter-efficient fine-tuning using Low-Rank Adaptation (LoRA) on a sentiment classification task.

## Overview
- **Task**: Binary sentiment classification on IMDb reviews
- **Base Model**: DistilBERT (66M parameters)
- **Adaptation**: LoRA with rank=8 (~0.1% additional parameters)
- **Goal**: Show how to achieve competitive accuracy with minimal parameter updates

## Step 1: Setup and Imports

In [ ]:
# Add src to path so we can import our modules
import sys
from pathlib import Path

# Change to project root
import os
os.chdir(Path.cwd() if 'notebooks' not in str(Path.cwd()) else Path.cwd().parent)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

import warnings
warnings.filterwarnings('ignore')

print("Setup complete! Working directory:", Path.cwd())

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Transformers and PEFT
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

# Our modules
from config import ExperimentConfig
from model import LoRAModel
from utils import set_seed, count_parameters, plot_training_history

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Check CUDA
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Load and Inspect Configuration

In [ ]:
# Load configuration from YAML
config = ExperimentConfig.from_yaml('configs/default_config.yaml')
print(config)

In [ ]:
# Set seed for reproducibility
set_seed(config.training.seed)

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 3: Initialize Model with LoRA

In [ ]:
# Initialize LoRA model
lora_model_wrapper = LoRAModel(config, device=device.type)
model, tokenizer = lora_model_wrapper.setup()

In [ ]:
# Analyze parameter efficiency
params = count_parameters(model)

print("\n" + "="*60)
print("Parameter Efficiency Analysis")
print("="*60)
print(f"Total Parameters (base model): {params['total']:,}")
print(f"Trainable Parameters (LoRA): {params['trainable']:,}")
print(f"Frozen Parameters: {params['frozen']:,}")
print(f"\nEfficiency Ratio: {params['efficiency_ratio']:.3f}%")
print(f"Reduction Factor: {params['total'] / params['trainable']:.0f}x")
print(f"\nMemory Savings (gradients only):")
print(f"  Full Fine-tune: ~{params['total'] * 4 / 1e6:.1f} MB")
print(f"  LoRA: ~{params['trainable'] * 4 / 1e6:.1f} MB")

## Step 4: Load and Prepare Dataset

In [ ]:
# Load IMDb dataset
print("Loading IMDb dataset...")
dataset = load_dataset('imdb', split='train')

# Limit to max_samples for quick iteration
if config.data.max_samples and len(dataset) > config.data.max_samples:
    dataset = dataset.select(range(config.data.max_samples))

print(f"Dataset size: {len(dataset)} samples")
print(f"Sample text: {dataset[0]['text'][:200]}...")
print(f"Label: {dataset[0]['label']} (0=negative, 1=positive)")

In [ ]:
# Tokenize dataset
def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=config.data.max_length
    )

print("Tokenizing dataset...")
dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
dataset = dataset.rename_column('label', 'labels')
dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenization complete!")
print(f"Sample tokenized: {dataset[0].keys()}")

In [ ]:
# Split into train/eval
from torch.utils.data import DataLoader, random_split

train_size = int(len(dataset) * config.data.train_test_split)
eval_size = len(dataset) - train_size

train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True
)
eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=config.training.batch_size
)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

## Step 5: Training Loop (Simplified)

Note: This is a quick demonstration. For full training, use `src/train.py`.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

def train_epoch(model, dataloader, optimizer, scheduler, device, num_steps=None):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    step_count = 0
    
    for batch in tqdm(dataloader, desc='Training', leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        step_count += 1
        
        # Optional: limit steps for quick demo
        if num_steps and step_count >= num_steps:
            break
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            predictions = logits.argmax(dim=-1)
            correct += (predictions == batch['labels']).sum().item()
            total += batch['labels'].size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy

print("Training functions defined!")

In [ ]:
# Setup optimizer and scheduler
optimizer = AdamW(
    model.parameters(),
    lr=config.training.learning_rate,
    weight_decay=config.training.weight_decay
)

total_steps = (len(train_dataloader) * config.training.epochs)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config.training.warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Optimizer: AdamW with lr={config.training.learning_rate}")

In [ ]:
# Training loop
print(f"\n{'='*60}")
print(f"Starting Training (Limited to 1 epoch for notebook demo)")
print(f"{'='*60}\n")

training_history = {
    'train_loss': [],
    'eval_loss': [],
    'eval_accuracy': []
}

# Train for 1 epoch (full training uses 3 epochs in train.py)
train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, device, num_steps=50)
training_history['train_loss'].append(train_loss)

eval_loss, eval_accuracy = evaluate(model, eval_dataloader, device)
training_history['eval_loss'].append(eval_loss)
training_history['eval_accuracy'].append(eval_accuracy)

print(f"\nEpoch 1/3 (partial):")
print(f"  Train Loss: {train_loss:.4f}")
print(f"  Eval Loss: {eval_loss:.4f}")
print(f"  Eval Accuracy: {eval_accuracy:.4f}")
print(f"\nNote: This is a quick demo with limited steps.")
print(f"For full training, run: python src/train.py --config configs/default_config.yaml")

## Step 6: Key Insights and Observations

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS: Why LoRA Works")
print("="*70)

insights = {
    "Parameter Efficiency": [
        "• LoRA reduces trainable parameters by 900x",
        "• Achieves this through low-rank matrix decomposition",
        "• ΔW = B·A where r=8 creates a bottleneck"
    ],
    "Memory Impact": [
        f"• Gradient storage: {params['trainable'] * 4 / 1e6:.1f} MB (vs {params['total'] * 4 / 1e6:.1f} MB)",
        "• Enables fine-tuning on consumer GPUs (8-16GB)",
        "• Reduces optimizer state memory significantly"
    ],
    "Why It Works": [
        "• Model adaptation lies in low-intrinsic-dimension subspace",
        "• Pre-trained models encode general knowledge that doesn't need updating",
        "• Task-specific adaptation is often 'thin' (low rank)"
    ],
    "Performance": [
        "• LoRA often matches or exceeds full fine-tuning",
        "• Avoids catastrophic forgetting (base weights frozen)",
        "• Produces modular, composable adapters"
    ]
}

for category, points in insights.items():
    print(f"\n{category}:")
    for point in points:
        print(f"  {point}")

## Step 7: Configuration Exploration

In [ ]:
# Show impact of different ranks
print("\nLoRA Rank Comparison (Parameter Count):")
print("-" * 60)

base_params = params['total']
hidden_dim = 768  # DistilBERT hidden dimension

rank_comparison = []
for r in [2, 4, 8, 16, 32]:
    # LoRA adds 2 * (hidden_dim * r) parameters per module
    # With 12 layers and 4 attention heads (Q, V, K, O), targeting Q and V
    num_modules = 12 * 2  # 12 layers, 2 modules per layer (Q, V)
    lora_params = num_modules * 2 * hidden_dim * r
    ratio = (lora_params / base_params) * 100
    reduction = base_params / lora_params
    
    rank_comparison.append({
        'Rank': r,
        'LoRA Params': f"{lora_params:,}",
        'Efficiency %': f"{ratio:.3f}%",
        'Reduction (x)': f"{reduction:.0f}x"
    })

df_ranks = pd.DataFrame(rank_comparison)
print(df_ranks.to_string(index=False))

print("\nTrade-offs:")
print("  r=4: Very lightweight, faster training, may underfit on complex tasks")
print("  r=8: Sweet spot for most tasks (what we use)")
print("  r=16+: Higher capacity, slower, approaches full fine-tuning cost")

## Step 8: Practical Considerations for Deployment

In [ ]:
print("\n" + "="*70)
print("PRACTICAL DEPLOYMENT CONSIDERATIONS")
print("="*70)

considerations = {
    "Model Serving": [
        "✓ Base model (66M) loaded once in GPU memory",
        "✓ Different LoRA adapters (~74K each) swapped per request",
        "→ One base model + multiple task-specific adapters = efficiency!"
    ],
    "Inference Optimization": [
        "→ LoRA weights can be merged into base model post-training",
        "→ No runtime overhead (merged W = W0 + B·A)",
        "→ Compatible with ONNX export and quantization"
    ],
    "Scaling to Larger Models": [
        "• 7B model + full fine-tune: ~28GB VRAM required",
        "• 7B model + LoRA (r=8): ~8GB VRAM required",
        "• 7B model + QLoRA (4-bit): ~4GB VRAM required"
    ],
    "When to Use Alternatives": [
        "• Prefix Tuning: When you need to keep input embeddings intact",
        "• Adapters: For more modular layer-wise adaptation",
        "• Full Fine-tune: When accuracy > efficiency (rare for LLMs)"
    ]
}

for title, items in considerations.items():
    print(f"\n{title}:")
    for item in items:
        print(f"  {item}")

## Summary: How to Use This Project

1. **For Quick Exploration**: Run this notebook (you are here!)
2. **For Full Training**: 
   ```bash
   python src/train.py --config configs/default_config.yaml
   ```
3. **For Evaluation**:
   ```bash
   python src/evaluate.py --model_path ./results/lora_model
   ```
4. **To Customize**: Edit `configs/default_config.yaml` to change model, hyperparameters, etc.

## Next Steps for Your Portfolio
- [ ] Run full training and document results
- [ ] Experiment with different `r` values and document findings
- [ ] Try a different model (e.g., BERT, GPT-2, Phi-2)
- [ ] Add QLoRA for larger models
- [ ] Compare LoRA with other efficient methods
- [ ] Add distributed training support